# Credit Scoring Model — Google Colab

**Objective:** Predict credit risk using Logistic Regression, Decision Tree, and Random Forest.

**Dataset:** UCI Statlog German Credit Data (1,000 records, 20 features). Official source: https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data

The original target is `1 = Good` and `2 = Bad`. This notebook maps **Bad risk to class 1** so Precision, Recall, F1 and ROC-AUC specifically evaluate detection of bad-risk records.

**Responsible use:** Educational project only; do not use the model alone to make real lending decisions.

## 1. Install dependencies

In [ ]:
%pip -q install ucimlrepo scikit-learn pandas numpy matplotlib seaborn joblib

## 2. Imports and project folders

In [ ]:
import os, json, joblib, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

PROJECT_DIR = "/content/Credit_Scoring_Model"
DATA_DIR = os.path.join(PROJECT_DIR, "dataset")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
for folder in [DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)
print("Project folder:", PROJECT_DIR)

## 3. Download the official UCI dataset

In [ ]:
dataset = fetch_ucirepo(id=144)
X_raw = dataset.data.features.copy()
y_raw = dataset.data.targets.copy()
print("Feature shape:", X_raw.shape, "| Target shape:", y_raw.shape)
print("Original target counts:")
display(y_raw.iloc[:, 0].value_counts())
display(X_raw.head())

## 4. Rename features and encode target

In [ ]:
feature_names = [
    "checking_account_status", "duration_months", "credit_history", "purpose",
    "credit_amount", "savings_account", "employment_since", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since", "property",
    "age_years", "other_installment_plans", "housing", "existing_credits",
    "job", "dependents", "telephone", "foreign_worker"
]
X = X_raw.copy()
if X.shape[1] == 20:
    X.columns = feature_names

target = y_raw.iloc[:, 0] if isinstance(y_raw, pd.DataFrame) else pd.Series(y_raw)
labels = set(pd.Series(target).dropna().unique().tolist())
print("Unique labels:", labels)
if labels.issubset({1, 2, 1.0, 2.0}):
    y = (pd.to_numeric(target) == 2).astype(int)  # Bad=1, Good=0
else:
    mapping = {"good": 0, "bad": 1, "1": 0, "2": 1}
    y = target.astype(str).str.lower().str.strip().map(mapping)
    if y.isna().any():
        raise ValueError("Unexpected target labels: " + str(labels))
y.name = "bad_risk"
print("Target counts (0=Good, 1=Bad):")
display(y.value_counts().sort_index())

## 5. Explore the data

In [ ]:
print("Rows and columns:", X.shape)
print("Missing values:", int(X.isna().sum().sum()))
display(X.describe(include="all").T)
plt.figure(figsize=(6,4))
sns.countplot(x=y.map({0:"Good risk", 1:"Bad risk"}))
plt.title("Credit risk class distribution")
plt.xlabel("Observed class")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

## 6. Feature engineering

The UCI dataset does not directly provide a current income or total-debt field. We will not invent those values. We add a simple derived feature, credit amount divided by duration. This is only a scale feature, not a real affordability or repayment calculation.

In [ ]:
X = X.copy()
if {"credit_amount", "duration_months"}.issubset(X.columns):
    duration = pd.to_numeric(X["duration_months"], errors="coerce").replace(0, np.nan)
    X["credit_amount_per_month"] = pd.to_numeric(X["credit_amount"], errors="coerce") / duration
display(X[["credit_amount", "duration_months", "credit_amount_per_month"]].head())

## 7. Split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 8. Preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
print("Numeric features:", numeric_features)
print("Categorical feature count:", len(categorical_features))

## 9. Train Logistic Regression, Decision Tree, Random Forest

In [ ]:
estimators = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
}
pipelines = {}
for name, estimator in estimators.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", estimator)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    print("Trained:", name)

## 10. Evaluate all models

Positive class = **Bad risk**. Accuracy is overall correctness; Precision measures how many predicted Bad-risk cases are truly Bad; Recall measures how many actual Bad-risk cases are found; F1 balances Precision and Recall; ROC-AUC evaluates ranking across thresholds. Actual values will appear when you run the notebook.

In [ ]:
rows, probabilities, predictions = [], {}, {}
for name, pipe in pipelines.items():
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    predictions[name], probabilities[name] = pred, prob
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (Bad risk)": precision_score(y_test, pred, zero_division=0),
        "Recall (Bad risk)": recall_score(y_test, pred, zero_division=0),
        "F1-score (Bad risk)": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })
results_df = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
display(results_df.style.format({c:"{:.3f}" for c in results_df.columns if c != "Model"}))
results_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison.csv"), index=False)

## 11. Plot metric comparison

In [ ]:
metric_cols = ["Accuracy", "Precision (Bad risk)", "Recall (Bad risk)", "F1-score (Bad risk)", "ROC-AUC"]
ax = results_df.set_index("Model")[metric_cols].plot(kind="bar", figsize=(12,6), ylim=(0,1.05))
ax.set_title("Credit scoring model comparison")
ax.set_ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "model_comparison.png"), dpi=150)
plt.show()

## 12. Classification reports and confusion matrices

In [ ]:
for name, pred in predictions.items():
    print("\\n" + "="*65 + "\\n" + name + "\\n" + "="*65)
    print(classification_report(y_test, pred, labels=[0,1],
          target_names=["Good risk", "Bad risk"], zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Predicted Good","Predicted Bad"],
        yticklabels=["Actual Good","Actual Bad"])
    plt.title("Confusion Matrix — " + name)
    plt.xlabel("Prediction"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix_" + name.lower().replace(" ","_") + ".png"), dpi=150)
    plt.show()

## 13. ROC curves

In [ ]:
plt.figure(figsize=(8,6))
for name, prob in probabilities.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=plt.gca())
plt.plot([0,1], [0,1], "--", label="Random baseline")
plt.title("ROC curves — Bad-risk detection")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_curves.png"), dpi=150)
plt.show()

## 14. Save the selected model

For this coursework demonstration, the model with the highest test ROC-AUC is saved. In rigorous model development, use a validation set or cross-validation to select models, and keep the test set untouched for final evaluation.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = pipelines[best_name]
model_path = os.path.join(MODEL_DIR, "best_credit_model.joblib")
joblib.dump(best_model, model_path)
metadata = {
    "selected_model_by_test_roc_auc": best_name,
    "target_mapping": {"0": "Good risk", "1": "Bad risk"},
    "features": X.columns.tolist(),
    "dataset": "UCI Statlog German Credit Data, ID 144",
    "note": "Educational use only; not for automated lending decisions."
}
with open(os.path.join(MODEL_DIR, "model_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print("Selected model:", best_name)
print("Saved:", model_path)

## 15. Try prediction and model confidence scores

In [ ]:
# Example: predict one held-out record, whose values match the dataset schema.
sample = X_test.iloc[[0]].copy()
predicted = int(best_model.predict(sample)[0])
proba = best_model.predict_proba(sample)[0]
display(sample.T)
print("Predicted label:", "Bad risk" if predicted == 1 else "Good risk")
print(f"Model score for Bad risk: {proba[1]:.1%}")
print(f"Model score for Good risk: {proba[0]:.1%}")
print("Actual held-out label:", "Bad risk" if int(y_test.iloc[0]) == 1 else "Good risk")
print("These scores are not guaranteed to be calibrated probabilities.")

## 16. Save dataset and bundle results for download

In [ ]:
export_df = X.copy()
export_df["bad_risk_target"] = y.values
export_df.to_csv(os.path.join(DATA_DIR, "german_credit_data_processed.csv"), index=False)

zip_path = "/content/Credit_Scoring_Model_artifacts.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_path[:-4], "zip", PROJECT_DIR)
print("Created:", zip_path)

# Uncomment to download the ZIP to your computer in Google Colab:
# from google.colab import files
# files.download(zip_path)

## 17. Responsible-use limitations

1. **Not a lending decision:** Never use this model alone to approve, reject, price, or limit real credit.
2. **Small historical sample:** The dataset has only 1,000 records and reflects an older context; results may not generalize to current applicants.
3. **Potential bias:** Attributes can encode or proxy for sensitive characteristics. Assess legality, fairness, and disparate impacts before any use.
4. **Metric trade-offs:** Accuracy can hide poor performance on the Bad-risk class. Review Precision, Recall, F1, ROC-AUC and confusion matrices together.
5. **Scores are not certainty:** `predict_proba` outputs may not be calibrated probabilities of default.
6. **Data privacy:** Use only lawfully collected data, minimize personal data, secure storage, and define retention rules.
7. **Human oversight:** Consequential use requires independent validation, explanations, appeal mechanisms and human review.
8. **No direct income/debt fields:** This dataset does not directly include current income or total debt. The derived ratio is not an affordability measure.

## 18. Conclusion template

After running the notebook, use the values in `results_df` to complete this paragraph:

“We trained Logistic Regression, Decision Tree, and Random Forest models on the UCI Statlog German Credit dataset. Preprocessing included imputation, numerical scaling, one-hot encoding, and a derived credit-amount-per-duration feature. We compared models using Accuracy, Precision, Recall, F1-score, ROC-AUC, classification reports, confusion matrices, and ROC curves. The highest ROC-AUC on this test split was achieved by **[model]**, with ROC-AUC **[value]**. The dataset is small and historical, so these results are educational and do not establish suitability for real lending decisions.”

**Citation:** Hofmann, H. (1994). *Statlog (German Credit Data)*. UCI Machine Learning Repository. https://doi.org/10.24432/C5NC77